# Bitcoin Sentiment × Trader Performance — ML Model

**Goal:** Predict whether a trade will be **profitable (Win/Loss)** using market sentiment features + trade context.

### Upload your files first
Run the cell below to upload `fear_greed_index.csv` and `historical_data.csv`.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload both CSVs here

## 1. Install & Import

In [ ]:
!pip install -q xgboost lightgbm shap imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

import xgboost as xgb
import lightgbm as lgb
import shap

print('All imports OK')

## 2. Load & Merge Data

In [ ]:
fg = pd.read_csv('fear_greed_index.csv')
hist = pd.read_csv('historical_data.csv')

# Parse dates
fg['date'] = pd.to_datetime(fg['date'])
hist['date'] = pd.to_datetime(
    hist['Timestamp IST'], format='%d-%m-%Y %H:%M', errors='coerce'
).dt.normalize()

# Merge on date
df = hist.merge(fg[['date', 'value', 'classification']], on='date', how='left')

print(f'Merged shape: {df.shape}')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
df.head(3)

## 3. Feature Engineering

In [ ]:
# ── Target variable ──────────────────────────────────────────────
# 1 = profitable trade (Closed PnL > 0), 0 = loss
# Only keep closed trades (PnL != 0)
df = df[df['Closed PnL'] != 0].copy()
df['target'] = (df['Closed PnL'] > 0).astype(int)
print(f'Class balance:\n{df["target"].value_counts(normalize=True).round(3)}')

# ── Sentiment features ───────────────────────────────────────────
df['fg_value'] = df['value'].fillna(df['value'].median())

sent_map = {'Extreme Fear': 0, 'Fear': 1, 'Neutral': 2, 'Greed': 3, 'Extreme Greed': 4}
df['fg_class_enc'] = df['classification'].map(sent_map).fillna(2)

# Is the trader going against the sentiment? (contrarian flag)
df['is_long'] = df['Direction'].str.startswith('Open Long').astype(int)
df['is_short'] = df['Direction'].str.startswith('Open Short').astype(int)
df['contrarian'] = (
    ((df['fg_class_enc'] <= 1) & (df['is_long'] == 1)) |   # Long in Fear
    ((df['fg_class_enc'] >= 3) & (df['is_short'] == 1))    # Short in Greed
).astype(int)

# ── Trade features ───────────────────────────────────────────────
df['side_enc'] = (df['Side'] == 'BUY').astype(int)
df['log_size_usd'] = np.log1p(df['Size USD'].abs().fillna(0))
df['log_fee'] = np.log1p(df['Fee'].abs().fillna(0))
df['exec_price'] = df['Execution Price'].fillna(df['Execution Price'].median())
df['start_pos'] = df['Start Position'].fillna(0)
df['crossed'] = df['Crossed'].astype(int)

# Direction encoding
dir_map = {
    'Open Long': 0, 'Close Long': 1,
    'Open Short': 2, 'Close Short': 3,
    'Buy': 4, 'Sell': 5
}
df['direction_enc'] = df['Direction'].map(dir_map).fillna(6)

# Coin encoding (top coins + 'other')
top_coins = df['Coin'].value_counts().head(10).index
df['coin_clean'] = df['Coin'].where(df['Coin'].isin(top_coins), other='OTHER')
le_coin = LabelEncoder()
df['coin_enc'] = le_coin.fit_transform(df['coin_clean'])

# ── Time features ────────────────────────────────────────────────
df['day_of_week'] = pd.to_datetime(df['date']).dt.dayofweek
df['month'] = pd.to_datetime(df['date']).dt.month

# ── Rolling sentiment momentum (3-day avg FG) ────────────────────
fg_roll = fg[['date','value']].set_index('date').sort_index()
fg_roll['fg_3d_avg'] = fg_roll['value'].rolling(3).mean()
fg_roll['fg_momentum'] = fg_roll['value'].diff(3)  # 3-day change in FG
df = df.merge(fg_roll[['fg_3d_avg','fg_momentum']].reset_index(), on='date', how='left')
df['fg_3d_avg'] = df['fg_3d_avg'].fillna(df['fg_value'])
df['fg_momentum'] = df['fg_momentum'].fillna(0)

print(f'\nFeature engineering done. Dataset size: {df.shape}')

## 4. Prepare Feature Matrix

In [ ]:
FEATURES = [
    # Sentiment
    'fg_value', 'fg_class_enc', 'fg_3d_avg', 'fg_momentum',
    # Trade context
    'side_enc', 'direction_enc', 'log_size_usd', 'log_fee',
    'exec_price', 'start_pos', 'crossed',
    # Derived
    'is_long', 'is_short', 'contrarian',
    'coin_enc', 'day_of_week', 'month'
]

X = df[FEATURES].fillna(0)
y = df['target']

print(f'Features: {X.shape[1]}  |  Samples: {X.shape[0]}')
print(f'Class balance — Win: {y.mean():.1%}  Loss: {(1-y.mean()):.1%}')

# Train / test split (time-aware: last 20% as test)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f'Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}')

## 5. Handle Class Imbalance with SMOTE

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f'After SMOTE — Win: {y_train_res.sum()}  Loss: {(y_train_res==0).sum()}')

## 6. Train Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=8,
                                             random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                   subsample=0.8, colsample_bytree=0.8,
                                   use_label_encoder=False, eval_metric='logloss',
                                   random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                     subsample=0.8, colsample_bytree=0.8,
                                     random_state=42, n_jobs=-1, verbose=-1)
}

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled  = scaler.transform(X_test)

results = {}

for name, model in models.items():
    print(f'Training {name}...', end=' ')
    X_tr = X_train_scaled if name == 'Logistic Regression' else X_train_res
    X_te = X_test_scaled  if name == 'Logistic Regression' else X_test
    model.fit(X_tr, y_train_res)
    preds = model.predict(X_te)
    proba = model.predict_proba(X_te)[:,1]
    auc = roc_auc_score(y_test, proba)
    results[name] = {'model': model, 'preds': preds, 'proba': proba, 'auc': auc}
    print(f'AUC = {auc:.4f}')

best_name = max(results, key=lambda k: results[k]['auc'])
print(f'\nBest model: {best_name} (AUC {results[best_name]["auc"]:.4f})')

## 7. Evaluation — Best Model

In [ ]:
best = results[best_name]

print(f'=== {best_name} — Classification Report ===')
print(classification_report(y_test, best['preds'], target_names=['Loss','Win']))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Confusion matrix
ConfusionMatrixDisplay(
    confusion_matrix(y_test, best['preds']),
    display_labels=['Loss','Win']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'{best_name} — Confusion Matrix')

# ROC curves (all models)
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    axes[1].plot(fpr, tpr, label=f"{name} ({res['auc']:.3f})")
axes[1].plot([0,1],[0,1],'k--',linewidth=0.8)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curves')
axes[1].legend(fontsize=8)

# AUC bar chart
names = list(results.keys())
aucs = [results[n]['auc'] for n in names]
colors = ['#378ADD' if n == best_name else '#B5D4F4' for n in names]
axes[2].barh(names, aucs, color=colors)
axes[2].set_xlim(0.5, 1.0)
axes[2].set_xlabel('AUC')
axes[2].set_title('Model Comparison')
for i, v in enumerate(aucs):
    axes[2].text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Feature Importance

In [ ]:
best_model = best['model']

if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=FEATURES)
    importances = importances.sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ['#378ADD' if v == importances.max() else
              '#1D9E75' if f.startswith('fg') else '#B5D4F4'
              for f, v in importances.items()]
    importances.plot(kind='barh', ax=ax, color=colors)
    ax.set_title(f'{best_name} — Feature Importance')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\nTop 5 features:')
    print(importances.sort_values(ascending=False).head())

## 9. SHAP Explainability

In [ ]:
# SHAP explains WHY the model makes each prediction
print('Computing SHAP values (may take ~30s)...')

X_shap = X_test.sample(min(2000, len(X_test)), random_state=42)

if best_name in ('XGBoost', 'LightGBM', 'Random Forest'):
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_shap)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # class 1 (Win)
else:
    explainer = shap.LinearExplainer(best_model, X_shap)
    shap_values = explainer.shap_values(X_shap)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap, feature_names=FEATURES,
                  show=False, plot_size=(10, 6))
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Sentiment-Stratified Performance Analysis

In [ ]:
# How well does the model do in each sentiment regime?
test_df = df.iloc[split_idx:].copy()
test_df['predicted'] = best['preds']
test_df['proba_win'] = best['proba']
test_df['correct'] = (test_df['predicted'] == test_df['target']).astype(int)

sent_perf = test_df.groupby('classification').agg(
    accuracy=('correct', 'mean'),
    avg_proba=('proba_win', 'mean'),
    trades=('correct', 'count'),
    actual_win_rate=('target', 'mean')
).round(3)

sent_order = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
sent_perf = sent_perf.reindex(sent_order).dropna()
print('Model accuracy per sentiment regime:')
print(sent_perf.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#E24B4A','#EF9F27','#888780','#1D9E75','#378ADD']

sent_perf['accuracy'].plot(kind='bar', ax=axes[0], color=colors, rot=15)
axes[0].set_title('Model Accuracy by Sentiment')
axes[0].set_ylabel('Accuracy')
axes[0].axhline(y=0.5, color='black', linestyle='--', linewidth=0.8, label='Baseline')
axes[0].legend()

sent_perf['actual_win_rate'].plot(kind='bar', ax=axes[1], color=colors, rot=15)
axes[1].set_title('Actual Win Rate by Sentiment')
axes[1].set_ylabel('Win Rate')

plt.tight_layout()
plt.savefig('sentiment_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Cross-Validation (Robustness Check)

In [ ]:
print('Running 5-fold stratified cross-validation on best model...')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X_train_res, y_train_res,
                             cv=cv, scoring='roc_auc', n_jobs=-1)
print(f'CV AUC scores: {[round(s,4) for s in cv_scores]}')
print(f'Mean: {cv_scores.mean():.4f}  ±  {cv_scores.std():.4f}')

## 12. Save Model & Download Results

In [ ]:
import pickle

# Save model + scaler + feature list
artifact = {
    'model': best_model,
    'scaler': scaler,
    'features': FEATURES,
    'label_encoder_coin': le_coin,
    'best_model_name': best_name,
    'test_auc': best['auc']
}

with open('trading_sentiment_model.pkl', 'wb') as f:
    pickle.dump(artifact, f)

print(f'Model saved: trading_sentiment_model.pkl')
print(f'Best model : {best_name}')
print(f'Test AUC   : {best["auc"]:.4f}')

# Download everything
from google.colab import files
for fname in ['trading_sentiment_model.pkl',
              'model_evaluation.png',
              'feature_importance.png',
              'shap_summary.png',
              'sentiment_accuracy.png']:
    try:
        files.download(fname)
    except:
        print(f'Could not download {fname}')

## 13. Inference — Predict on a New Trade

In [ ]:
# Example: predict win probability for a new trade
def predict_trade(fg_value, direction, side, size_usd, fee,
                  exec_price, start_pos, crossed, coin,
                  fg_3d_avg=None, fg_momentum=0, day_of_week=0, month=1):
    sent_map = {'Extreme Fear': 0, 'Fear': 1, 'Neutral': 2, 'Greed': 3, 'Extreme Greed': 4}
    if fg_value <= 25: fg_class = 0
    elif fg_value <= 45: fg_class = 1
    elif fg_value <= 55: fg_class = 2
    elif fg_value <= 75: fg_class = 3
    else: fg_class = 4

    dir_map = {'Open Long':0,'Close Long':1,'Open Short':2,'Close Short':3,'Buy':4,'Sell':5}
    is_long = 1 if direction == 'Open Long' else 0
    is_short = 1 if direction == 'Open Short' else 0
    contrarian = int((fg_class <= 1 and is_long) or (fg_class >= 3 and is_short))

    coin_clean = coin if coin in le_coin.classes_ else 'OTHER'
    coin_enc = le_coin.transform([coin_clean])[0]

    row = pd.DataFrame([{
        'fg_value': fg_value,
        'fg_class_enc': fg_class,
        'fg_3d_avg': fg_3d_avg or fg_value,
        'fg_momentum': fg_momentum,
        'side_enc': 1 if side == 'BUY' else 0,
        'direction_enc': dir_map.get(direction, 6),
        'log_size_usd': np.log1p(abs(size_usd)),
        'log_fee': np.log1p(abs(fee)),
        'exec_price': exec_price,
        'start_pos': start_pos,
        'crossed': int(crossed),
        'is_long': is_long,
        'is_short': is_short,
        'contrarian': contrarian,
        'coin_enc': coin_enc,
        'day_of_week': day_of_week,
        'month': month
    }])

    proba = best_model.predict_proba(row[FEATURES])[0][1]
    return {'win_probability': round(proba, 4), 'prediction': 'WIN' if proba > 0.5 else 'LOSS'}

# Example prediction
result = predict_trade(
    fg_value=20,          # Extreme Fear
    direction='Open Long',
    side='BUY',
    size_usd=5000,
    fee=2.5,
    exec_price=95000,
    start_pos=0,
    crossed=False,
    coin='BTC',
    day_of_week=1,
    month=3
)
print('Prediction for BTC Long in Extreme Fear:')
print(result)